# 🔢 06 - Sorting and Filtering in Pandas

Notebook 03 introduced boolean masks and `.loc[]`. This notebook puts them to work: sorting, building long filters you can still read, `.query()`, searching text, and turning a question into a sorted report.

## ✅ Learning Goals
- Sort rows with `.sort_values()`, including by multiple columns, and know that sorting returns a **new** DataFrame
- Filter data with one or more conditions, using `&`, `|`, and `~`
- Keep long filters readable with **named masks**
- Use `.query()`, including `in`, `@variables`, and column names with spaces
- Search text columns with `.str.contains()` (including patterns like `'son|ez'`), `.str.startswith()`, and `.str.endswith()`
- Answer a question with a filtered, sorted report

> **The core habit:** filter with clear conditions, sort only after selecting the rows you need, and use `.query()` or named masks when an expression gets long.

## 1. Load the Dataset

In [1]:
import pandas as pd

df = pd.read_csv('students.csv')
df.head()

,first_name,last_name,math_score,science_score
0,Danielle,Wood,100,95
1,Angel,Clark,67,78
2,Joshua,Adams,61,100
3,Jeffrey,Zuniga,77,99
4,Jill,Wong,75,83


## 2. Sorting Data

In [2]:
# Sort by math_score, ascending (the default)
df.sort_values(by='math_score')

,first_name,last_name,math_score,science_score
28,Melanie,Baker,60,94
2,Joshua,Adams,61,100
19,Adam,Adams,61,70
13,James,Cohen,61,66
12,Anthony,Hawkins,62,74
44,Christopher,Brown,62,64
14,Debra,Cameron,65,84
9,Anthony,Gray,65,74
37,Christine,Hicks,65,80
49,Michael,Valencia,65,91


In [3]:
# Sort by science_score, descending
df.sort_values(by='science_score', ascending=False)

,first_name,last_name,math_score,science_score
17,Bridget,Johnson,92,100
2,Joshua,Adams,61,100
34,Nicholas,Ross,73,100
26,Christopher,Rocha,97,100
3,Jeffrey,Zuniga,77,99
25,Matthew,Foster,88,98
46,Craig,Ferrell,94,96
5,Erica,Lynch,74,96
0,Danielle,Wood,100,95
35,Andrea,Robinson,81,95


In [4]:
# Sort by multiple columns — science_score descending, then math_score ascending
# to break ties
df.sort_values(by=['science_score', 'math_score'], ascending=[False, True])

,first_name,last_name,math_score,science_score
2,Joshua,Adams,61,100
34,Nicholas,Ross,73,100
17,Bridget,Johnson,92,100
26,Christopher,Rocha,97,100
3,Jeffrey,Zuniga,77,99
25,Matthew,Foster,88,98
5,Erica,Lynch,74,96
46,Craig,Ferrell,94,96
35,Andrea,Robinson,81,95
0,Danielle,Wood,100,95


The **order** of the `by=` list sets the priority: sort by the first column, and use the second only to break **ties** in the first. `ascending=` takes a matching list, one direction per column.

`.sort_values()` returns a **new**, sorted DataFrame. The original keeps its order unless you assign the result:

In [5]:
by_math = df.sort_values('math_score')            # lowest math score first
print('sorted copy starts with:', by_math['first_name'].iloc[0])
print('df still starts with:   ', df['first_name'].iloc[0])

sorted copy starts with: Melanie
df still starts with:    Danielle


💡 For "the top 5" or "the bottom 3" of one column, `df.nlargest(5, 'math_score')` and `df.nsmallest(3, 'math_score')` are shortcuts (notebook 03).

## 3. Filtering with Conditions

In [6]:
# Who scored at least 90 in math?
df[df['math_score'] >= 90]

,first_name,last_name,math_score,science_score
0,Danielle,Wood,100,95
8,Robert,Martin,94,62
10,Jeffery,Mayo,97,78
17,Bridget,Johnson,92,100
18,Lisa,Stewart,98,83
20,Linda,Clark,95,83
22,Matthew,Walter,94,73
26,Christopher,Rocha,97,100
42,Jeremy,Conner,98,85
46,Craig,Ferrell,94,96


In [7]:
# Who scored less than 70 in either subject?
df[(df['math_score'] < 70) | (df['science_score'] < 70)]

,first_name,last_name,math_score,science_score
1,Angel,Clark,67,78
2,Joshua,Adams,61,100
6,Patricia,Jackson,68,72
7,Christopher,Daniel,66,64
8,Robert,Martin,94,62
9,Anthony,Gray,65,74
11,William,Bowman,87,65
12,Anthony,Hawkins,62,74
13,James,Cohen,61,66
14,Debra,Cameron,65,84


## 4. Combining Filters
Recap from notebook 03: `&` means **and**, `|` means **or**, `~` means **not**, and every condition goes in its own parentheses. (Python's `and`/`or` keywords raise a `ValueError` here.)

In [8]:
# Students with 87+ in both subjects
df[(df['math_score'] >= 87) & (df['science_score'] >= 87)]

,first_name,last_name,math_score,science_score
0,Danielle,Wood,100,95
17,Bridget,Johnson,92,100
25,Matthew,Foster,88,98
26,Christopher,Rocha,97,100
46,Craig,Ferrell,94,96


### Named Masks for Long Filters
A filter with three or four conditions turns into one long, hard-to-read line. Instead, give each condition a **name** first, then combine the names. It reads like a sentence, and you can check each mask on its own:

In [9]:
strong_math = df['math_score'] >= 85
strong_science = df['science_score'] >= 85
late_alphabet = df['last_name'] >= 'M'           # text compares alphabetically

print('strong in math:', strong_math.sum())      # check each piece separately
df[strong_math & strong_science & late_alphabet]

strong in math: 15


,first_name,last_name,math_score,science_score
0,Danielle,Wood,100,95
26,Christopher,Rocha,97,100


`.isin()`, `.between()`, and `~` (notebook 03) all combine the same way. For example, `~df['last_name'].isin(['Smith', 'Jones'])` is "everyone **except** Smith and Jones".

## 5. Using `.query()` for Readability

In [10]:
# The same filter, written with .query()
df.query('math_score >= 87 and science_score >= 87')

,first_name,last_name,math_score,science_score
0,Danielle,Wood,100,95
17,Bridget,Johnson,92,100
25,Matthew,Foster,88,98
26,Christopher,Rocha,97,100
46,Craig,Ferrell,94,96


Inside `.query()`, you write `and`, `or`, and `not` (not `&`, `|`, `~`), and you can test membership with `in`:

In [11]:
df.query("last_name in ['Wood', 'Wong', 'Clark']")

,first_name,last_name,math_score,science_score
0,Danielle,Wood,100,95
1,Angel,Clark,67,78
4,Jill,Wong,75,83
20,Linda,Clark,95,83


### Using Python Variables with `@`
Put `@` in front of a **Python variable** to use its value. Without the `@`, pandas looks for a **column** with that name:

In [12]:
minimum = 90
df.query('math_score >= @minimum and science_score >= @minimum')

,first_name,last_name,math_score,science_score
0,Danielle,Wood,100,95
17,Bridget,Johnson,92,100
26,Christopher,Rocha,97,100
46,Craig,Ferrell,94,96


### Column Names with Spaces
`.query()` can't read a name like `Unit Price` directly. Wrap it in **backticks**. Better still, standardize the names first (notebook 05), and the problem goes away:

In [13]:
sales = pd.DataFrame({'Product Name': ['Notebook', 'Pen', 'Lamp'], 'Unit Price': [4.99, 1.50, 24.99]})
print(sales.query('`Unit Price` < 5'))

sales.columns = sales.columns.str.strip().str.lower().str.replace(' ', '_')
sales.query('unit_price < 5')

  Product Name  Unit Price
0     Notebook        4.99
1          Pen        1.50


,product_name,unit_price
0,Notebook,4.99
1,Pen,1.50


**Masks or `.query()`?** Both are good. Use whichever reads more clearly to the next person. `.query()` shines for long numeric conditions, and masks are better for text methods like `.str.contains()`.

## 6. Searching Text with `.str`
Remember the string methods from Module 02? A text column's `.str` gives you those same methods, applied to **every** value at once, and the result works as a filter mask:

In [14]:
df[df['last_name'].str.startswith('W')]

,first_name,last_name,math_score,science_score
0,Danielle,Wood,100,95
4,Jill,Wong,75,83
22,Matthew,Walter,94,73
29,Cassandra,Wright,70,75
39,Barbara,Williams,66,74


In [15]:
df[df['last_name'].str.endswith('son')]

,first_name,last_name,math_score,science_score
6,Patricia,Jackson,68,72
17,Bridget,Johnson,92,100
35,Andrea,Robinson,81,95
43,Keith,Henderson,76,77


In [16]:
# 'an' anywhere in the name, ignoring capitalization
df[df['first_name'].str.contains('an', case=False, na=False)]

,first_name,last_name,math_score,science_score
0,Danielle,Wood,100,95
1,Angel,Clark,67,78
9,Anthony,Gray,65,74
12,Anthony,Hawkins,62,74
24,Susan,Rios,74,64
28,Melanie,Baker,60,94
29,Cassandra,Wright,70,75
32,Amanda,Richards,77,84
35,Andrea,Robinson,81,95
40,Janet,Rodriguez,82,62


- `case=False` matches `Angel` and `Danielle` alike, whatever the capitalization.
- `na=False` treats missing names as "no match". Without it, a single `NaN` in the column makes the filter fail.

Other string methods work the same way: `.str.upper()`, `.str.strip()`, `.str.len()`, `.str.replace()`, and more.

### Patterns: `|` Means "Or" Inside `.str.contains()`
`.str.contains()` treats its search text as a **regular expression** (a pattern), where `|` means "or":

In [17]:
df[df['last_name'].str.contains('son|ez', na=False)]

,first_name,last_name,math_score,science_score
6,Patricia,Jackson,68,72
15,Amy,Parsons,73,77
16,Jeffrey,Perez,74,89
17,Bridget,Johnson,92,100
35,Andrea,Robinson,81,95
40,Janet,Rodriguez,82,62
43,Keith,Henderson,76,77


That's powerful, but it means some characters, like `+`, `.`, `(`, and `|`, have **special** meanings. Searching for a literal `C++` quietly matches any `C`. Turn patterns off with `regex=False`:

In [18]:
books = pd.Series(['C++ Primer', 'C Programming', 'Python Basics'])
print('as a pattern:', books.str.contains('C++').tolist())                # 'C Programming' matches too!
print('literal text:', books.str.contains('C++', regex=False).tolist())

as a pattern: [True, True, False]
literal text: [True, False, False]


### Case-Insensitive Starts and Ends
`.str.startswith()` and `.str.endswith()` have no `case=` option. Lowercase the column first, then compare with lowercase text:

In [19]:
df[df['first_name'].str.lower().str.startswith('a')]

,first_name,last_name,math_score,science_score
1,Angel,Clark,67,78
9,Anthony,Gray,65,74
12,Anthony,Hawkins,62,74
15,Amy,Parsons,73,77
19,Adam,Adams,61,70
32,Amanda,Richards,77,84
35,Andrea,Robinson,81,95


### Combining Text Searches with Other Conditions
A text mask combines with number masks like any other mask, and named masks keep it readable:

In [20]:
ends_in_y = df['first_name'].str.lower().str.endswith('y')
passed_math = df['math_score'] >= 70

df[ends_in_y & passed_math]

,first_name,last_name,math_score,science_score
3,Jeffrey,Zuniga,77,99
10,Jeffery,Mayo,97,78
15,Amy,Parsons,73,77
16,Jeffrey,Perez,74,89
30,Lindsay,Hoffman,87,70
42,Jeremy,Conner,98,85


## 7. Filter, Then Sort: Building a Report
Most real questions combine everything: **filter** the rows, pick the **columns**, then **sort**. Filter first, so you only sort the rows you actually need:

In [21]:
df.loc[df['science_score'] >= 90, ['first_name', 'last_name', 'science_score']].sort_values(
    'science_score', ascending=False)

,first_name,last_name,science_score
2,Joshua,Adams,100
34,Nicholas,Ross,100
26,Christopher,Rocha,100
17,Bridget,Johnson,100
3,Jeffrey,Zuniga,99
25,Matthew,Foster,98
46,Craig,Ferrell,96
5,Erica,Lynch,96
0,Danielle,Wood,95
35,Andrea,Robinson,95


### A Complete Walkthrough
**Question:** *Which students scored at least 80 in **both** subjects and have a last name that ends in "son" or "ez", or starts with "W"? Rank them by their average.*

Break the question into named masks, combine them, then build the report:

In [22]:
both_80 = (df['math_score'] >= 80) & (df['science_score'] >= 80)
name_match = df['last_name'].str.contains('son|ez', na=False) | df['last_name'].str.startswith('W', na=False)

report = df[both_80 & name_match].copy()
report['average'] = (report['math_score'] + report['science_score']) / 2
report = report[['first_name', 'last_name', 'math_score', 'science_score', 'average']].sort_values(
    'average', ascending=False)
report

,first_name,last_name,math_score,science_score,average
0,Danielle,Wood,100,95,97.5
17,Bridget,Johnson,92,100,96.0
35,Andrea,Robinson,81,95,88.0


`.copy()` makes it clear that `report` is a separate table, so adding the `average` column doesn't touch `df` (notebook 03). The everyday pattern is:

```python
df.loc[row_condition, ['column_a', 'column_b']].sort_values('column_a')
```

## 🏋️ Practice

### Practice 1 — Alphabetical Sort (easy)
Sort the dataset by `first_name`, alphabetically.

In [31]:
# Your code here
df.sort_values(by='first_name', ascending=True, inplace=True)
df['first_name'].head(10) # displays the first 10 names after sorting.

19       Adam
32     Amanda
15        Amy
35     Andrea
1       Angel
12    Anthony
9     Anthony
39    Barbara
41    Brandon
17    Bridget
Name: first_name, dtype: str

### Practice 2 — Close Scores (medium)
Find students whose `math_score` and `science_score` are within **2 points** of each other. Hint: `(df['math_score'] - df['science_score']).abs() <= 2` makes the mask. Sort the result by `last_name`.

In [35]:
# Your code here
# Create the mask
score_diff = (df['math_score'] - df['science_score']).abs() <= 2

# Filter, then sort
result = df[score_diff].sort_values(by='last_name')

result

,first_name,last_name,math_score,science_score
44,Christopher,Brown,62,64
41,Brandon,Burton,82,80
7,Christopher,Daniel,66,64
46,Craig,Ferrell,94,96
43,Keith,Henderson,76,77


### Practice 3 — Query with OR (harder)
Use `.query()` to find students who scored over 95 in *either* subject (hint: `.query()` uses `or`, not `|`).

In [38]:
# Your code here
df.query('math_score >= 95 | science_score >= 95')

,first_name,last_name,math_score,science_score
2,Joshua,Adams,61,100
20,Linda,Clark,95,83
42,Jeremy,Conner,98,85
46,Craig,Ferrell,94,96
25,Matthew,Foster,88,98
17,Bridget,Johnson,92,100
5,Erica,Lynch,74,96
10,Jeffery,Mayo,97,78
35,Andrea,Robinson,81,95
26,Christopher,Rocha,97,100


### Practice 4 — Search a Bigger Dataset (medium)
Load `pokemon.csv`. Find every Pokémon whose `Name` contains `'mega'`, regardless of capitalization, and sort them by `Total`, highest first. How many are there?

In [45]:
poke = pd.read_csv('pokemon.csv')
# Your code here
poke
players = poke[poke['Name'].str.contains('mega', case=False, na=False)]
players.sort_values(by='Total', ascending=False)

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
163,150,MewtwoMega Mewtwo X,Psychic,Fighting,780,106,190,100,154,100,130,1,True
164,150,MewtwoMega Mewtwo Y,Psychic,NaN,780,106,150,70,194,120,140,1,True
426,384,RayquazaMega Rayquaza,Dragon,Flying,780,105,180,100,180,100,115,3,True
420,381,LatiosMega Latios,Dragon,Psychic,700,80,130,100,160,120,110,3,True
413,376,MetagrossMega Metagross,Steel,Psychic,700,80,145,150,105,110,110,3,False
409,373,SalamenceMega Salamence,Dragon,Flying,700,95,145,130,120,90,120,3,False
796,719,DiancieMega Diancie,Rock,Fairy,700,50,160,110,160,110,110,6,True
494,445,GarchompMega Garchomp,Dragon,Ground,700,108,170,115,120,95,92,4,False
268,248,TyranitarMega Tyranitar,Rock,Dark,700,100,164,150,95,120,71,2,False
418,380,LatiasMega Latias,Dragon,Psychic,700,80,100,120,140,150,110,3,True


In [46]:
players.sort_values(by='Total', ascending=False).head(10) #displaying the top 10 Pokémon with "mega" in their name, sorted by Total score in descending order.

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
163,150,MewtwoMega Mewtwo X,Psychic,Fighting,780,106,190,100,154,100,130,1,True
164,150,MewtwoMega Mewtwo Y,Psychic,NaN,780,106,150,70,194,120,140,1,True
426,384,RayquazaMega Rayquaza,Dragon,Flying,780,105,180,100,180,100,115,3,True
420,381,LatiosMega Latios,Dragon,Psychic,700,80,130,100,160,120,110,3,True
413,376,MetagrossMega Metagross,Steel,Psychic,700,80,145,150,105,110,110,3,False
409,373,SalamenceMega Salamence,Dragon,Flying,700,95,145,130,120,90,120,3,False
796,719,DiancieMega Diancie,Rock,Fairy,700,50,160,110,160,110,110,6,True
494,445,GarchompMega Garchomp,Dragon,Ground,700,108,170,115,120,95,92,4,False
268,248,TyranitarMega Tyranitar,Rock,Dark,700,100,164,150,95,120,71,2,False
418,380,LatiasMega Latias,Dragon,Psychic,700,80,100,120,140,150,110,3,True


### Practice 5 — Query with Variables (medium)
Create two variables, `min_math` and `min_science`. Use `.query()` with `@` to find students who meet **both** minimums, sorted by `math_score` from highest to lowest. Then change the variables and re-run, without touching the query itself.

In [51]:
min_math = 90
min_science = 90

students_min = (
    df.query('math_score >= @min_math and science_score >= @min_science')
      .sort_values(by='math_score', ascending=False)
)

students_min

,first_name,last_name,math_score,science_score
0,Danielle,Wood,100,95
26,Christopher,Rocha,97,100
46,Craig,Ferrell,94,96
17,Bridget,Johnson,92,100


In [52]:
# Your code here
min_math = 85
min_science = 89

students_min = (
    df.query('math_score >= @min_math and science_score >= @min_science')
      .sort_values(by='math_score', ascending=False)
)

students_min

,first_name,last_name,math_score,science_score
0,Danielle,Wood,100,95
26,Christopher,Rocha,97,100
46,Craig,Ferrell,94,96
17,Bridget,Johnson,92,100
25,Matthew,Foster,88,98


## ✏️ Your Turn
1. Sort the dataset by `last_name`, then by `first_name` to break ties.
2. Use named masks to find students with a `math_score` from 70 to 85 (`.between()`) whose `first_name` contains `'e'` (any capitalization).
3. Use `.query()` to select rows matching a condition of your own choosing.
4. Add an `average_score` column, sort by it descending, and show just the top 5 students' names and averages.

In [56]:
# Your turn here
# 1. Sort by last_name, then first_name
sorted_df = df.sort_values(by=['last_name', 'first_name'])
print(sorted_df.head())

   first_name last_name  math_score  science_score
19       Adam     Adams          61             70
2      Joshua     Adams          61            100
31    Jessica     Baker          81             89
28    Melanie     Baker          60             94
11    William    Bowman          87             65


In [ ]:
# 2. Named masks
math_sorted = (df['math_score'].between(70, 85))
first_name_sort = df['first_name'].str.contains('e', na=False, case=False)
result =df[math_sorted & first_name_sort]
print(result)

   first_name  last_name  math_score  science_score
35     Andrea   Robinson          81             95
5       Erica      Lynch          74             96
21      Helen       Hahn          72             82
40      Janet  Rodriguez          82             62
16    Jeffrey      Perez          74             89
3     Jeffrey     Zuniga          77             99
31    Jessica      Baker          81             89
43      Keith  Henderson          76             77
38   Michelle      Davis          84             63


In [61]:
# 3. Use .query() with your own condition
# Example: students with math_score >= 90 and science_score >= 90

high_achievers = df.query('math_score >= 90 and science_score >= 90')
print(high_achievers)

     first_name last_name  math_score  science_score
17      Bridget   Johnson          92            100
26  Christopher     Rocha          97            100
46        Craig   Ferrell          94             96
0      Danielle      Wood         100             95


In [62]:
# 4. Add average_score, sort descending, show top 5

df['average_score'] = (df['math_score'] + df['science_score']) / 2

top5 = (
    df.sort_values('average_score', ascending=False)
      [['first_name', 'last_name', 'average_score']]
      .head(5)
)

print(top5)

     first_name last_name  average_score
26  Christopher     Rocha           98.5
0      Danielle      Wood           97.5
17      Bridget   Johnson           96.0
46        Craig   Ferrell           95.0
25      Matthew    Foster           93.0


## 🔥 Challenge (Optional) — A Product Report
Using the product table below, answer: *Which Office and Travel products cost less than \$30, have at least 10 in stock, and have `'note'` or `'pen'` in their name (any capitalization)?*

1. Build a **named mask** for each condition, and combine them.
2. Show only `product_name`, `department`, `price`, `stock`, and `rating`, sorted by `department` A–Z, then `price` from highest to lowest.
3. Answer the same question again with `.query()` for the number conditions, plus a mask for the name. Which version is easier to read?

In [63]:
products = pd.DataFrame({
    'product_id':   [101, 102, 103, 104, 105, 106],
    'product_name': ['Notebook', 'Travel Backpack', 'Blue Pen', 'Desk Lamp', 'Water Bottle', 'Notebook Stand'],
    'department':   ['Office', 'Travel', 'Office', 'Home', 'Travel', 'Office'],
    'price':        [4.99, 39.99, 1.50, 24.99, 12.50, 29.99],
    'stock':        [32, 14, 120, 8, 25, 11],
    'rating':       [4.6, 4.8, 4.2, 4.5, 4.7, 4.3],
})
# Your code here


In [ ]:
# Masks for each condition
dept_mask = products['department'].isin(['Office', 'Travel'])
price_mask = products['price'] < 30
stock_mask = products['stock'] >= 10
name_mask = products['product_name'].str.contains(
    'note|pen', case=False, na=False
)

# Combine all masks
filtered_products = products[
    dept_mask & price_mask & stock_mask & name_mask
]

# Select columns and sort
result = (filtered_products[['product_name', 'department', 'price', 'stock', 'rating']].sort_values(
        by=['department', 'price'],
        ascending=[True, False]))

print(result)

     product_name department  price  stock  rating
5  Notebook Stand     Office  29.99     11     4.3
0        Notebook     Office   4.99     32     4.6
2        Blue Pen     Office   1.50    120     4.2


In [65]:
name_mask = products['product_name'].str.contains('note|pen', case=False, na=False)

result_query = (
    products.query(
        "department in ['Office', 'Travel'] and price < 30 and stock >= 10"
    )[name_mask]
    [['product_name', 'department', 'price', 'stock', 'rating']]
    .sort_values(
        by=['department', 'price'],
        ascending=[True, False]
    )
)

print(result_query)

     product_name department  price  stock  rating
5  Notebook Stand     Office  29.99     11     4.3
0        Notebook     Office   4.99     32     4.6
2        Blue Pen     Office   1.50    120     4.2


C:\Users\Nazmun\AppData\Local\Temp\ipykernel_25068\1227179518.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  products.query(


- masking metod is easier to read and cleaner then query method. 

---
### 📝 Summary
| Task              | Code Example                                 |
|-------------------|------------------------------------------------|
| Sort by column     | `df.sort_values(by='col')` (returns a new DataFrame) |
| Sort by several    | `df.sort_values(by=['a', 'b'], ascending=[True, False])` |
| Filter condition   | `df[df['col'] > value]`                       |
| Combine filters    | `df[(cond1) & (cond2)]` (and), or, `~` (not)   |
| Named masks        | `is_x = ...; is_y = ...; df[is_x & is_y]`      |
| Query method       | `df.query('col > value and col2 < value')`    |
| Query with variables | `df.query('col >= @minimum')`               |
| Query with spaces  | ``df.query('`Unit Price` < 5')``              |
| Search text        | `df[df['col'].str.contains('x', case=False, na=False)]` |
| Pattern / literal  | `.str.contains(pattern)`, add `regex=False` for literal text |
| Filter, then sort  | `df.loc[cond, cols].sort_values('col')`       |